# WeatherForYnov — LSTM Prévision Température Mensuelle

**Hackathon YNOV — Sujet 2 : Prévisions météorologiques**

Ce notebook entraîne un modèle **LSTM multivarié** pour prédire la **température moyenne sur les 12 prochains mois** à partir des données `climat_source.csv` (2001–2022).

## Paramètres du modèle
| Paramètre | Valeur |
|-----------|--------|
| Périmètre | Toutes stations (modèle global poolé) |
| Entrée | 120 mois (10 ans) × N features météo |
| Sortie | 12 mois de `temp_moy_mensuelle` |
| Train | 2001–2020 |
| Test | 2021–2022 |
| Architecture | LSTM(64) + Dropout(0.2) + Dense(12) |

## Métriques d'évaluation
- **MAE** — Mean Absolute Error (°C)
- **RMSE** — Root Mean Squared Error (°C)
- **MAPE** — Mean Absolute Percentage Error (%)
- **R²** — Coefficient de détermination

---
## 1. Configuration et imports

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 100

# Reproductibilité
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Chemins
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SOURCE_CSV = ROOT / "src" / "data" / "processed" / "climat_source.csv"
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparamètres
WINDOW      = 120   # 10 ans d'historique en entrée
HORIZON     = 12    # 12 mois à prédire
TRAIN_END   = 2020  # inclus
TEST_START  = 2021
TEST_END    = 2022
BATCH_SIZE  = 64
MAX_EPOCHS  = 150

# Cible
TARGET = "temp_moy_mensuelle"

print(f"TensorFlow version : {tf.__version__}")
print(f"Source CSV         : {SOURCE_CSV}")
print(f"Fenêtre d'entrée   : {WINDOW} mois | Horizon de prédiction : {HORIZON} mois")

---
## 2. Chargement et préparation des données

In [ ]:
df_raw = pd.read_csv(SOURCE_CSV, parse_dates=["date"])
print(f"Dimensions brutes : {df_raw.shape}")
print(f"Colonnes          : {list(df_raw.columns)}")
print(f"Période           : {df_raw['annee'].min()} – {df_raw['annee'].max()}")
print(f"Nb stations       : {df_raw['NUM_POSTE'].nunique()}")
display(df_raw.head(3))

In [ ]:
# Filtrage 2001-2022
df = df_raw[(df_raw["annee"] >= 2001) & (df_raw["annee"] <= TEST_END)].copy()

# Features météo numériques
FEATURES = [
    "temp_moy_mensuelle",
    "temp_max",
    "temp_min",
    "precipitations_mm",
    "humidite",
    "pression_mer",
    "vent_moyen",
    "insolation",
    "rayonnement_global",
    "evapotranspiration",
]
# Garder uniquement les features disponibles dans le CSV
FEATURES = [f for f in FEATURES if f in df.columns]
print(f"Features retenues : {FEATURES}")

# Sélection des colonnes utiles
df = df[["NUM_POSTE", "nom_station", "annee", "mois", "date"] + FEATURES].copy()

# Trier par station puis par date
df = df.sort_values(["NUM_POSTE", "annee", "mois"]).reset_index(drop=True)
print(f"\nDimensions après filtre : {df.shape}")
display(df.head())

In [ ]:
# Analyse des valeurs manquantes
missing = df[FEATURES].isnull().mean().sort_values(ascending=False) * 100
print("Taux de valeurs manquantes (%) :")
display(missing.to_frame("% NaN").round(2))

# Imputation par interpolation linéaire par station
df[FEATURES] = df.groupby("NUM_POSTE")[FEATURES].transform(
    lambda x: x.interpolate(method="linear", limit_direction="both")
)

# Remplissage résiduel par médiane globale de la feature
for col in FEATURES:
    df[col] = df[col].fillna(df[col].median())

print(f"\nValeurs manquantes restantes : {df[FEATURES].isnull().sum().sum()}")

In [ ]:
# Garder uniquement les stations avec la série complète nécessaire
# Minimum requis : WINDOW + HORIZON mois = 132 mois en train
MIN_MONTHS_TRAIN = WINDOW + HORIZON  # 132 mois minimum
FULL_SERIES_LENGTH = (TEST_END - 2001 + 1) * 12  # 264 mois

station_counts = df.groupby("NUM_POSTE")["date"].count()
valid_stations = station_counts[
    station_counts >= MIN_MONTHS_TRAIN + HORIZON
].index.tolist()

df = df[df["NUM_POSTE"].isin(valid_stations)].copy()
print(f"Stations valides (>= {MIN_MONTHS_TRAIN + HORIZON} mois) : {len(valid_stations)}")
print(f"Lignes conservées : {len(df)}")

---
## 3. Feature Engineering

In [ ]:
# Encodage cyclique du mois (capture la saisonnalité)
df["mois_sin"] = np.sin(2 * np.pi * df["mois"] / 12)
df["mois_cos"] = np.cos(2 * np.pi * df["mois"] / 12)

# Lag features par station (température des mois précédents)
for lag in [1, 3, 6, 12]:
    df[f"temp_lag_{lag}"] = df.groupby("NUM_POSTE")["temp_moy_mensuelle"].shift(lag)

# Rolling stats par station (moyenne et écart-type glissants)
df["temp_roll12_mean"] = (
    df.groupby("NUM_POSTE")["temp_moy_mensuelle"]
    .transform(lambda x: x.rolling(window=12, min_periods=1).mean())
)
df["temp_roll12_std"] = (
    df.groupby("NUM_POSTE")["temp_moy_mensuelle"]
    .transform(lambda x: x.rolling(window=12, min_periods=1).std().fillna(0))
)

# Remplissage des NaN créés par les lags
lag_cols = [f"temp_lag_{lag}" for lag in [1, 3, 6, 12]]
df[lag_cols] = df.groupby("NUM_POSTE")[lag_cols].transform(
    lambda x: x.fillna(method="bfill").fillna(method="ffill")
)

# Liste complète des features d'entrée du modèle
ALL_FEATURES = (
    FEATURES
    + ["mois_sin", "mois_cos"]
    + lag_cols
    + ["temp_roll12_mean", "temp_roll12_std"]
)

N_FEATURES = len(ALL_FEATURES)
print(f"Nombre total de features : {N_FEATURES}")
print(f"Features : {ALL_FEATURES}")

---
## 4. Normalisation et création des séquences

In [ ]:
# Normalisation par station (MinMaxScaler [-1, 1] pour chaque station)
# Ajustement du scaler UNIQUEMENT sur les données d'entraînement
scalers = {}  # scalers[NUM_POSTE] = {"X": scaler_X, "y": scaler_y}
df_scaled_list = []

for poste, grp in df.groupby("NUM_POSTE"):
    grp = grp.sort_values("date").copy()

    train_mask = grp["annee"] <= TRAIN_END

    # Scaler des features
    scaler_X = MinMaxScaler(feature_range=(-1, 1))
    scaler_X.fit(grp.loc[train_mask, ALL_FEATURES])

    # Scaler de la cible (pour l'inverse transform lors de l'évaluation)
    scaler_y = MinMaxScaler(feature_range=(-1, 1))
    scaler_y.fit(grp.loc[train_mask, [TARGET]])

    scalers[poste] = {"X": scaler_X, "y": scaler_y}

    # Application de la normalisation
    grp[ALL_FEATURES] = scaler_X.transform(grp[ALL_FEATURES])
    df_scaled_list.append(grp)

df_scaled = pd.concat(df_scaled_list, ignore_index=True)
print(f"Données normalisées : {df_scaled.shape}")
print(f"Scalers créés pour {len(scalers)} stations")

In [ ]:
def create_sequences(group_df, window, horizon, features, target):
    """Crée les séquences X (window × features) et y (horizon valeurs cible) pour une station."""
    X, y = [], []
    values = group_df[features].values
    targets = group_df[target].values

    for i in range(len(values) - window - horizon + 1):
        X.append(values[i : i + window])
        y.append(targets[i + window : i + window + horizon])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []
test_meta = []  # pour garder la trace station / date de début prédiction

for poste, grp in df_scaled.groupby("NUM_POSTE"):
    grp = grp.sort_values("date").reset_index(drop=True)

    # Données d'entraînement
    train_grp = grp[grp["annee"] <= TRAIN_END].copy()
    if len(train_grp) >= WINDOW + HORIZON:
        X_tr, y_tr = create_sequences(train_grp, WINDOW, HORIZON, ALL_FEATURES, TARGET)
        X_train_list.append(X_tr)
        y_train_list.append(y_tr)

    # Données de test : on utilise la fenêtre des WINDOW derniers mois du train
    # + les premiers mois du test pour évaluer les prédictions 2021-2022
    full_grp = grp.copy()
    test_grp = full_grp[full_grp["annee"] >= TEST_START - (WINDOW // 12)].copy()
    if len(test_grp) >= WINDOW + HORIZON:
        X_te, y_te = create_sequences(test_grp, WINDOW, HORIZON, ALL_FEATURES, TARGET)
        # Garder seulement les séquences dont la prédiction porte sur 2021-2022
        for j in range(len(X_te)):
            pred_start_idx = test_grp.index[WINDOW + j] if (WINDOW + j) < len(test_grp) else None
            if pred_start_idx is not None:
                pred_year = test_grp.loc[pred_start_idx, "annee"] if pred_start_idx in test_grp.index else None
                if pred_year is not None and pred_year >= TEST_START:
                    X_test_list.append(X_te[j])
                    y_test_list.append(y_te[j])
                    test_meta.append({"NUM_POSTE": poste})
                    break  # 1 séquence de test par station

X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)
X_test  = np.array(X_test_list,  dtype=np.float32)
y_test  = np.array(y_test_list,  dtype=np.float32)

print(f"X_train : {X_train.shape}  (séquences × fenêtre × features)")
print(f"y_train : {y_train.shape}  (séquences × horizon)")
print(f"X_test  : {X_test.shape}")
print(f"y_test  : {y_test.shape}")

In [ ]:
# Mélange aléatoire du jeu d'entraînement
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(X_train))
X_train, y_train = X_train[idx], y_train[idx]

# Validation split (15% du train)
val_size = int(0.15 * len(X_train))
X_val, y_val = X_train[:val_size], y_train[:val_size]
X_train, y_train = X_train[val_size:], y_train[val_size:]

print(f"Train      : {X_train.shape[0]} séquences")
print(f"Validation : {X_val.shape[0]} séquences")
print(f"Test       : {X_test.shape[0]} séquences (1 par station)")

---
## 5. Architecture du modèle LSTM

In [ ]:
def build_lstm_model(window, n_features, horizon, lstm_units=64, dropout_rate=0.2):
    """LSTM léger : 1 couche LSTM (64) + Dropout + Dense de sortie."""
    inputs = keras.Input(shape=(window, n_features), name="input_seq")

    x = layers.LSTM(lstm_units, return_sequences=False, name="lstm_1")(inputs)
    x = layers.Dropout(dropout_rate, name="dropout_1")(x)
    x = layers.Dense(32, activation="relu", name="dense_hidden")(x)
    outputs = layers.Dense(horizon, activation="linear", name="output")(x)

    model = keras.Model(inputs, outputs, name="LSTM_TempForecast")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="huber",
        metrics=["mae"],
    )
    return model


model = build_lstm_model(
    window=WINDOW,
    n_features=N_FEATURES,
    horizon=HORIZON,
    lstm_units=64,
    dropout_rate=0.2,
)
model.summary()

---
## 6. Entraînement

In [ ]:
MODEL_PATH = MODELS_DIR / "lstm_temp_best.keras"

cb_list = [
    callbacks.EarlyStopping(
        monitor="val_loss", patience=20, restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=10, min_lr=1e-6, verbose=1
    ),
    callbacks.ModelCheckpoint(
        filepath=str(MODEL_PATH), monitor="val_loss", save_best_only=True, verbose=0
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=cb_list,
    verbose=1,
)

print(f"\nModèle sauvegardé : {MODEL_PATH}")
print(f"Meilleure val_loss : {min(history.history['val_loss']):.4f}")

---
## 7. Courbes d'apprentissage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss (Huber)
axes[0].plot(history.history["loss"],     label="Train loss",      linewidth=2)
axes[0].plot(history.history["val_loss"], label="Validation loss", linewidth=2, linestyle="--")
axes[0].set_title("Courbe de perte (Huber Loss)")
axes[0].set_xlabel("Époque")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# MAE
axes[1].plot(history.history["mae"],     label="Train MAE",      linewidth=2)
axes[1].plot(history.history["val_mae"], label="Validation MAE", linewidth=2, linestyle="--")
axes[1].set_title("Courbe MAE (normalisée)")
axes[1].set_xlabel("Époque")
axes[1].set_ylabel("MAE")
axes[1].legend()
axes[1].grid(True)

plt.suptitle("Courbes d'apprentissage — LSTM Température", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(MODELS_DIR / "learning_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Graphique sauvegardé : {MODELS_DIR / 'learning_curves.png'}")

---
## 8. Évaluation sur le jeu de test (2021–2022)

In [ ]:
def mape(y_true, y_pred):
    """Mean Absolute Percentage Error — évite la division par zéro."""
    mask = np.abs(y_true) > 0.1
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


# Prédictions sur le jeu de test (espace normalisé)
y_pred_scaled = model.predict(X_test, verbose=0)

# Dénormalisation : inverse transform par station
y_pred_list, y_true_list = [], []

for i, meta in enumerate(test_meta):
    poste = meta["NUM_POSTE"]
    if poste not in scalers:
        continue
    scaler_y = scalers[poste]["y"]

    pred_raw = y_pred_scaled[i].reshape(-1, 1)
    true_raw = y_test[i].reshape(-1, 1)

    pred_inv = scaler_y.inverse_transform(pred_raw).flatten()
    true_inv = scaler_y.inverse_transform(true_raw).flatten()

    y_pred_list.append(pred_inv)
    y_true_list.append(true_inv)

y_pred_all = np.concatenate(y_pred_list)
y_true_all = np.concatenate(y_true_list)

# Calcul des métriques
mae  = mean_absolute_error(y_true_all, y_pred_all)
rmse = np.sqrt(mean_squared_error(y_true_all, y_pred_all))
r2   = r2_score(y_true_all, y_pred_all)
mape_val = mape(y_true_all, y_pred_all)

print("=" * 50)
print("   MÉTRIQUES D'ÉVALUATION — TEST 2021-2022")
print("=" * 50)
print(f"  MAE   : {mae:.3f} °C")
print(f"  RMSE  : {rmse:.3f} °C")
print(f"  MAPE  : {mape_val:.2f} %")
print(f"  R²    : {r2:.4f}")
print("=" * 50)

# Métriques par horizon (mois 1 à 12)
mae_by_horizon  = [mean_absolute_error(y_true_list[i], y_pred_list[i]) for i in range(len(y_true_list))]
print(f"\nMAE moyenne par station : {np.mean(mae_by_horizon):.3f} °C")

In [ ]:
# Métriques par pas de temps (mois 1 à 12)
mae_per_step  = []
rmse_per_step = []

y_pred_matrix = np.array(y_pred_list)  # (n_stations, 12)
y_true_matrix = np.array(y_true_list)  # (n_stations, 12)

for step in range(HORIZON):
    mae_per_step.append(mean_absolute_error(y_true_matrix[:, step], y_pred_matrix[:, step]))
    rmse_per_step.append(np.sqrt(mean_squared_error(y_true_matrix[:, step], y_pred_matrix[:, step])))

fig, ax = plt.subplots(figsize=(10, 4))
x = range(1, HORIZON + 1)
ax.plot(x, mae_per_step,  marker="o", label="MAE",  linewidth=2)
ax.plot(x, rmse_per_step, marker="s", label="RMSE", linewidth=2)
ax.set_xlabel("Pas de temps (mois à l'avance)")
ax.set_ylabel("Erreur (°C)")
ax.set_title("Erreur de prédiction par horizon (1 à 12 mois)")
ax.set_xticks(list(x))
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig(MODELS_DIR / "error_by_horizon.png", dpi=120, bbox_inches="tight")
plt.show()

---
## 9. Visualisation : prédit vs réel

In [ ]:
# Sélection de 6 stations pour la visualisation
N_DISPLAY = min(6, len(test_meta))
indices = np.linspace(0, len(test_meta) - 1, N_DISPLAY, dtype=int)

months_labels = [f"M+{i+1}" for i in range(HORIZON)]

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharey=False)
axes = axes.flatten()

for ax, idx in zip(axes, indices):
    poste = test_meta[idx]["NUM_POSTE"]
    station_name = df.loc[df["NUM_POSTE"] == poste, "nom_station"].iloc[0] if poste in df["NUM_POSTE"].values else str(poste)

    ax.plot(months_labels, y_true_list[idx], marker="o", label="Réel",    linewidth=2, color="steelblue")
    ax.plot(months_labels, y_pred_list[idx], marker="s", label="Prédit",  linewidth=2, color="tomato",    linestyle="--")

    mae_s  = mean_absolute_error(y_true_list[idx], y_pred_list[idx])
    ax.set_title(f"{station_name}\nMAE = {mae_s:.2f} °C", fontsize=9)
    ax.set_ylabel("Temp. moy. (°C)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.4)
    ax.tick_params(axis="x", rotation=45)

plt.suptitle(
    "Température mensuelle — Prédit vs Réel (2021–2022, 6 stations)",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig(MODELS_DIR / "pred_vs_real.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Scatter plot global : réel vs prédit
fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(y_true_all, y_pred_all, alpha=0.3, s=8, color="steelblue")

lims = [min(y_true_all.min(), y_pred_all.min()) - 1,
        max(y_true_all.max(), y_pred_all.max()) + 1]
ax.plot(lims, lims, "r--", linewidth=1.5, label="y = x (parfait)")

ax.set_xlabel("Température réelle (°C)")
ax.set_ylabel("Température prédite (°C)")
ax.set_title(f"Réel vs Prédit — tous horizons\nMAE={mae:.2f}°C | RMSE={rmse:.2f}°C | R²={r2:.3f}")
ax.legend()
ax.grid(True, alpha=0.4)
ax.set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.savefig(MODELS_DIR / "scatter_pred_vs_real.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Distribution des résidus
residuals = y_pred_all - y_true_all

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(residuals, bins=60, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--", linewidth=1.5)
axes[0].set_title("Distribution des résidus")
axes[0].set_xlabel("Résidu (°C) = Prédit − Réel")
axes[0].set_ylabel("Fréquence")

axes[1].scatter(y_true_all, residuals, alpha=0.2, s=6, color="steelblue")
axes[1].axhline(0, color="red", linestyle="--", linewidth=1.5)
axes[1].set_title("Résidus vs Valeurs réelles")
axes[1].set_xlabel("Température réelle (°C)")
axes[1].set_ylabel("Résidu (°C)")

plt.suptitle("Analyse des résidus", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(MODELS_DIR / "residuals.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Résidu moyen (biais) : {residuals.mean():.3f} °C")
print(f"Écart-type résidus   : {residuals.std():.3f} °C")

---
## 10. Récapitulatif final

In [ ]:
summary = pd.DataFrame({
    "Métrique": ["MAE (°C)", "RMSE (°C)", "MAPE (%)", "R²"],
    "Valeur":   [f"{mae:.3f}", f"{rmse:.3f}", f"{mape_val:.2f}", f"{r2:.4f}"],
    "Interprétation": [
        "Erreur moyenne en degré Celsius",
        "Erreur quadratique moyenne (pénalise les grands écarts)",
        "Erreur relative moyenne en pourcentage",
        "Proportion de variance expliquée (1.0 = parfait)",
    ],
})

print("\n" + "=" * 60)
print("        SYNTHÈSE FINALE DU MODÈLE LSTM")
print("=" * 60)
print(f"  Architecture  : LSTM(64) + Dropout(0.2) + Dense(12)")
print(f"  Fenêtre input : {WINDOW} mois (10 ans)")
print(f"  Horizon       : {HORIZON} mois")
print(f"  Train         : 2001–{TRAIN_END} | Test : {TEST_START}–{TEST_END}")
print(f"  Nb features   : {N_FEATURES}")
print("=" * 60)
display(summary)
print(f"\nModèle enregistré : {MODEL_PATH}")